In [2]:
#!pip install google-cloud-aiplatform vertexai pandas python-dotenv

In [3]:
import os
import json
import time
from typing import Dict, Any, Tuple

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

In [4]:
load_dotenv()

VERTEX_PROJECT = os.getenv("VERTEX_PROJECT", "etui-cloud-architecture")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "europe-west4")
VERTEX_MODEL   = os.getenv("VERTEX_MODEL", "gemini-1.5-pro-002")

INPUT_CSV  = "labelling_full_human.csv"
OUTPUT_CSV = None  # if None, will save as "<input>_gemini.csv"

MAX_CHARS = 12000  # truncate overly long chunks safely

GEN_CONFIG = GenerationConfig(
    temperature=0.1,
    top_p=0.2,
    top_k=32,
    max_output_tokens=512,
    response_mime_type="application/json",  # strong hint for JSON
)

print("Project:", VERTEX_PROJECT)
print("Location:", VERTEX_LOCATION)
print("Model:", VERTEX_MODEL)

Project: etui-cloud-architecture
Location: europe-west4
Model: gemini-1.5-pro-002


## Initialize Vertex AI & model

In [5]:
vertexai.init(project=VERTEX_PROJECT, location=VERTEX_LOCATION)
model = GenerativeModel(VERTEX_MODEL)
model

## 5) Prompt template (clear rubric + JSON contract)

In [6]:
PROMPT_TEMPLATE = """You are classifying whether a policy text *talks about the value of research data*.

Return ONLY JSON with keys: "label" (yes|no|unsure), "explanation" (<= 3 sentences, objective), "evidence" (array with up to 2 short verbatim spans from the text OR an empty array if the signal is implicit).

## Task
Given the TEXT, decide if it expresses that *research data has value*. Value can be explicit or implicit.

### Consider "value" broadly, including:
- economic/financial/commercial value (e.g., asset, IP, competitive advantage, monetization)
- intrinsic/scientific value (e.g., enables knowledge creation, reuse, verification, reproducibility)
- societal/public value (e.g., public good, societal impact, innovation, policy-making)
- strategic/organizational value (e.g., strategic asset, efficiency gains, institutional advantage)
- utility-based value (e.g., useful, impactful, increases quality of research, accelerates discovery)

### Positive cues (any one is sufficient):
- Direct statements that data *has value*, *is valuable*, *creates value/impact/benefit*.
- Data framed as an asset, capital, resource with *benefits/returns*.
- Mentions of *reuse, sharing, openness, FAIR* explicitly linked to *benefits/impact/value*.
- Statements that data management/curation *creates competitive edge*, *supports innovation*, or *yields societal benefits*.

### Implicit cues (acceptable as YES if clearly implied):
- Describing data as foundational for research progress, verification, or collaboration *in a way that attributes benefit or impact to the data itself*, not just to compliance.
- Policy goals emphasizing outcomes that are *benefits derived from data* (e.g., increased impact, reuse) beyond mere procedures.

### Negative cues (lean NO if dominant):
- Purely procedural or compliance language (roles, approvals, storage locations, retention schedules) with no claim of value/benefit/impact.
- Mentions of "importance" of *processes* (e.g., good RDM practice) without linking that importance to *value of the data itself*.

### UNSURE:
- Vague wording about "importance" or "good practice" where it is not clear whether *data itself* is considered to have value.
- Conflicting signals or insufficient context.

### Output format (JSON ONLY):
{{
  "label": "yes|no|unsure",
  "explanation": "brief reason referencing the text (max 3 sentences).",
  "evidence": ["short quote 1", "short quote 2"]
}}

TEXT:
\"\"\"{text}\"
\"\"\"
"""

## 6) Helper functions (prompt build, retries, normalization)

In [7]:
def build_prompt(text: str) -> str:
    cleaned = (text or "").strip()
    if len(cleaned) > MAX_CHARS:
        cleaned = cleaned[:MAX_CHARS] + " …"
    return PROMPT_TEMPLATE.format(text=cleaned)

def call_gemini_json(model: GenerativeModel, prompt: str, retries: int = 5, base_delay: float = 1.0) -> Dict[str, Any]:
    """
    Calls Gemini with structured JSON expectations and retries on transient errors.
    """
    last_err = None
    for attempt in range(retries):
        try:
            resp = model.generate_content(prompt, generation_config=GEN_CONFIG)
            raw = resp.text  # JSON string expected
            return json.loads(raw)
        except Exception as e:
            last_err = e
            time.sleep(base_delay * (2 ** attempt) + 0.1 * attempt)
    raise RuntimeError(f"Gemini call failed after {retries} attempts: {last_err}")

def normalize_output(data: Dict[str, Any]) -> Tuple[str, str]:
    """
    Ensure label ∈ {yes,no,unsure} and produce a compact explanation including short evidence if present.
    """
    label = str(data.get("label", "")).strip().lower()
    if label not in {"yes", "no", "unsure"}:
        label = "unsure"

    explanation = (data.get("explanation") or "").strip()
    evidence = data.get("evidence") or []
    if isinstance(evidence, list) and evidence:
        ev_join = " | ".join([str(x)[:200] for x in evidence[:2]])
        if ev_join and ev_join not in explanation:
            explanation = (explanation + f" Evidence: {ev_join}").strip()

    explanation = " ".join(explanation.split())  # compact whitespace
    return label, explanation

## 7) Load data, sanity checks

In [8]:
df = pd.read_csv(INPUT_CSV)
assert "text" in df.columns, "Input CSV must contain a 'text' column."

# Prepare output columns (idempotent)
if "gemini_class" not in df.columns:
    df["gemini_class"] = pd.NA
if "gemini_explanation" not in df.columns:
    df["gemini_explanation"] = pd.NA

df.head(3)

,name,chunk_id,heading_path,section_label,text,num_tokens,num_sentences,answer,explanation,gemini_class,gemini_explanation
0,aalto-university.md,0,Aalto University > Aalto University Research D...,NaN,The research data management policy aims to ma...,76,3,yes,Explicit: 'impact' appears near 'data'.,<NA>,<NA>
1,aalto-university.md,1,Aalto University > Aalto University Research D...,NaN,The data management policy shall be implemente...,87,4,no,No: no explicit or implicit value-of-data cues...,<NA>,<NA>
2,aalto-university.md,2,Aalto University > Aalto University Research D...,NaN,Ownership of copyright protected research data...,71,3,no,No: no explicit or implicit value-of-data cues...,<NA>,<NA>


## 8) (Optional) Dry-run parameter

In [9]:
DRY_RUN_N = 0  # set >0 to process only the first N rows (e.g., 20)
process_n = len(df) if DRY_RUN_N <= 0 else min(DRY_RUN_N, len(df))
print("Rows to process:", process_n)

Rows to process: 2649


## 9) Classify rows

In [ ]:
processed = 0
for idx in tqdm(range(process_n), desc="Classifying"):
    row = df.iloc[idx]
    # Skip already-labeled rows to allow safe restarts
    if pd.notna(row.get("gemini_class")) and str(row.get("gemini_class")).strip():
        continue
    text = row.get("text", "")
    try:
        payload = call_gemini_json(model, build_prompt(text))
        label, explanation = normalize_output(payload)
    except Exception as e:
        label, explanation = "unsure", f"Automatic classification failed: {e}"

    df.at[df.index[idx], "gemini_class"] = label
    df.at[df.index[idx], "gemini_explanation"] = explanation
    processed += 1

print(f"Newly processed rows: {processed}")
df.head(3)

## 10) Save results (non-destructive)

In [ ]:
if OUTPUT_CSV is None:
    base, ext = os.path.splitext(INPUT_CSV)
    OUTPUT_CSV = f"{base}_gemini{ext or '.csv'}"

df.to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV

## 11) (Optional) Lightweight QC: distribution & quick peeks

In [ ]:
display(df["gemini_class"].value_counts(dropna=False))
df.sample(5)[["text","gemini_class","gemini_explanation"]]